# Exploration : ponctualité ferroviaire en France

Questions métier :
1. **Quelles liaisons sont les moins ponctuelles**, et est-ce structurel ou ponctuel ?
2. **Quelle saisonnalité ?** Quels mois sont les plus difficiles ?
3. **D'où viennent les retards ?** Quelle part relève de la SNCF, quelle part est externe ?
4. **La météo a-t-elle un effet mesurable** une fois la saisonnalité retirée ?

Données : tables `marts` produites par dbt (voir `transform/`).

In [1]:
import duckdb, pandas as pd, plotly.express as px
pd.set_option("display.float_format", "{:.3f}".format)
con = duckdb.connect("../data/ponctualite.duckdb", read_only=True)
q = lambda s: con.execute(s).df()

## 1. Liaisons les moins ponctuelles (TGV, 2023-2025)

Seuil de 1 000 trains pour éviter les petites liaisons à forte variance.

In [2]:
pires = q('''
    select l.libelle_liaison, l.type_service, sum(f.nb_trains_circules) trains,
           1 - sum(f.nb_trains_retard_arrivee)/sum(f.nb_trains_circules) ponctualite,
           stddev(f.taux_ponctualite) volatilite_mensuelle,
           sum(f.nb_trains_retard_sup_60)/sum(f.nb_trains_circules) part_retard_sup_60
    from marts.fct_regularite_liaison_mois f join marts.dim_liaison l using (liaison_id)
    where f.activite='TGV' and year(f.mois) between 2023 and 2025 and not f.est_incoherent
    group by all having sum(f.nb_trains_circules) >= 1000 order by ponctualite''')
pires.head(10)

,libelle_liaison,type_service,trains,ponctualite,volatilite_mensuelle,part_retard_sup_60
0,Paris Est → Stuttgart,International,4337.000,0.670,0.242,0.032
1,Italie → Paris Gare de Lyon,International,1846.000,0.697,0.100,0.054
2,Lyon Part-Dieu → Lille,National,11785.000,0.719,0.059,0.040
3,Lyon Part-Dieu → Marne-la-Vallée Chessy,National,12287.000,0.731,0.052,0.031
4,Mâcon Loché → Paris Gare de Lyon,National,7797.000,0.743,0.077,0.031
5,Chambéry Challes-les-Eaux → Paris Gare de Lyon,National,8254.000,0.752,0.068,0.034
6,Paris Est → Francfort,International,5316.000,0.766,0.118,0.029
7,Lyon Part-Dieu → Marseille Saint-Charles,National,17088.000,0.770,0.041,0.039
8,Marseille Saint-Charles → Lille,National,7425.000,0.775,0.053,0.041
9,Arras → Paris Nord,National,17026.000,0.782,0.043,0.014


In [3]:
# Structurel ou conjoncturel ? Rang de chaque liaison d'une année sur l'autre.
rangs = q('''
    select liaison_id, year(mois) annee, 1 - sum(nb_trains_retard_arrivee)/sum(nb_trains_circules) p
    from marts.fct_regularite_liaison_mois
    where activite='TGV' and not est_incoherent and year(mois) between 2022 and 2025
    group by all having sum(nb_trains_circules) >= 300''').pivot(index="liaison_id", columns="annee", values="p")
rangs.corr(method="spearman")

annee,2022,2023,2024,2025
annee,,,,
2022,1.000,0.872,0.796,0.870
2023,0.872,1.000,0.854,0.873
2024,0.796,0.854,1.000,0.894
2025,0.870,0.873,0.894,1.000


**Lecture** : une corrélation de rang élevée d'une année sur l'autre signifie que les liaisons
peu ponctuelles le restent : le problème est **structurel** (axe saturé, longueur du trajet, infrastructure),
pas un accident d'une année.

## 2. Saisonnalité

In [4]:
saison = q('''
    select n.activite, d.num_mois, d.nom_mois, avg(n.taux_ponctualite) ponctualite
    from marts.mart_ponctualite_nationale_mois n join marts.dim_date d using (mois)
    where not d.est_periode_covid group by all order by num_mois''')
px.line(saison, x="nom_mois", y="ponctualite", color="activite", markers=True,
        title="Ponctualité moyenne par mois calendaire (hors Covid)").update_yaxes(tickformat=".0%")

In [5]:
saison.pivot(index='nom_mois', columns='activite', values='ponctualite').sort_values('TGV').head(4)

activite,Intercités,TER,TGV
nom_mois,,,
juillet,0.824,0.909,0.808
juin,0.811,0.904,0.816
novembre,0.810,0.886,0.838
décembre,0.837,0.902,0.848


## 3. Causes des retards (TGV, retards > 15 min)

In [6]:
causes = q('''
    select year(f.mois) annee, c.responsable, sum(f.nb_trains_retard_imputes) trains
    from marts.fct_causes_retard_mois f join marts.dim_cause c using (code_cause)
    group by all''')
part = causes.pivot(index="annee", columns="responsable", values="trains")
part.div(part.sum(axis=1), axis=0)

responsable,Gares & Connexions / Transporteur,Hors SNCF,SNCF Réseau,Transporteur
annee,,,,
2018,0.051,0.274,0.433,0.242
2019,0.062,0.262,0.459,0.217
2020,0.059,0.276,0.468,0.197
2021,0.083,0.267,0.331,0.320
2022,0.085,0.196,0.399,0.320
2023,0.095,0.174,0.436,0.296
2024,0.089,0.180,0.452,0.279
2025,0.088,0.209,0.448,0.255
2026,0.186,0.310,0.220,0.283


**Lecture** : environ un quart des retards est externe (météo, obstacles, malveillance, grèves).
Les trois quarts restants relèvent du système ferroviaire : SNCF Réseau (infrastructure, gestion du trafic)
et transporteur (matériel, gestion en gare, voyageurs). C'est là que se trouvent les leviers d'action.

⚠️ **Rupture de série en mai 2026** : à partir de ce mois, les parts « gestion du trafic » et « infrastructure »
chutent d'un coup et « gestion en gare » triple. Un tel saut en un mois signale presque certainement un changement
de nomenclature d'imputation chez la source, pas une évolution réelle. Les comparaisons de causes s'arrêtent donc à avril 2026.

## 4. Effet de la météo

Piège classique : la ponctualité et la météo ont toutes deux une **saisonnalité**. Corréler les séries brutes
reviendrait à mesurer l'effet « été » (chaleur + travaux + affluence) plutôt que l'effet météo.

Méthode : pour chaque région et chaque mois calendaire, on calcule l'**écart** à la ponctualité habituelle de ce
mois dans cette région, puis on le compare à la météo du mois. On compare un juillet chaud à un juillet normal.

In [7]:
ter = q('''
    select f.region, f.mois, month(f.mois) m, f.taux_ponctualite,
           f.meteo_jours_gel gel, f.meteo_jours_chaleur chaleur,
           f.meteo_precipitations_mm pluie, f.meteo_jours_pluie_forte pluie_forte, f.meteo_jours_vent_fort vent
    from marts.fct_regularite_ter_mois f join marts.dim_date d using (mois)
    where not d.est_periode_covid''')
grp = ter.groupby(["region", "m"])
ter["ecart_ponct"] = ter.taux_ponctualite - grp.taux_ponctualite.transform("mean")
for v in ["gel", "chaleur", "pluie", "pluie_forte", "vent"]:
    ter[f"anomalie_{v}"] = ter[v] - grp[v].transform("mean")
ter.filter(like="anomalie").corrwith(ter.ecart_ponct).sort_values().to_frame("corrélation avec l'écart de ponctualité")

,corrélation avec l'écart de ponctualité
anomalie_chaleur,-0.220
anomalie_pluie_forte,-0.144
anomalie_pluie,-0.140
anomalie_vent,-0.140
anomalie_gel,-0.040


In [8]:
# Effet concret : mois avec beaucoup plus de jours de pluie forte que d'habitude (quartile supérieur)
q75 = ter.anomalie_pluie_forte.quantile(0.75)
ter.assign(mois_tres_pluvieux=ter.anomalie_pluie_forte > q75).groupby("mois_tres_pluvieux").ecart_ponct.agg(["mean", "count"])

,mean,count
mois_tres_pluvieux,,
False,0.002,1221
True,-0.005,407


In [9]:
px.scatter(ter, x="anomalie_chaleur", y="ecart_ponct", trendline="ols", opacity=0.3,
           title="TER : jours de chaleur en plus vs écart de ponctualité",
           labels={"anomalie_chaleur": "Jours ≥ 30 °C en plus que la normale", "ecart_ponct": "Écart de ponctualité"}
).update_yaxes(tickformat="+.1%")

## Conclusions

- **Les liaisons les moins ponctuelles le restent d'une année sur l'autre** : le problème est structurel, donc la priorisation des investissements peut s'appuyer sur l'historique.
- **L'été et la fin d'automne sont les périodes les plus dures**, toutes activités confondues.
- **Environ 3 retards sur 4 relèvent du système ferroviaire** (infrastructure, gestion du trafic, matériel). La météo et les autres causes externes n'expliquent qu'environ un quart des retards.
- **L'effet météo est réel mais modeste** une fois la saisonnalité neutralisée : un mois nettement plus chaud que la normale est le signal le plus net (corrélation −0,22), devant la pluie forte et le vent ; le gel n'a quasiment pas d'effet. La météo seule explique une petite part de la variance, ce qui est cohérent avec la répartition des causes.
- **Qualité des données** : rupture de nomenclature des causes en mai 2026, doublons Intercités (relations jour/nuit), lignes incohérentes en avril 2020. Tous sont traités ou signalés dans dbt.
- Pour la **prévision** (voir `ml/forecast.py`), l'historique récent de la liaison est de loin la variable la plus utile.